# Lateralized HCFIM EEG Analysis V1

This notebook extends the reproducible HCFIM EEG paper analysis with a
lateralized cortical schema. It is intended as a companion experiment to
`01_hcfim_eeg_paper_reproducible.ipynb`.

Analytical scope:

1. Define `lateralized_v1` cortical regions.
2. Extract PSD and HCFIM features by lateralized region and frequency band.
3. Aggregate epoch-level features to one row per subject.
4. Add right-minus-left lateralization deltas.
5. Rank HCFIM geometries by intergroup separability and intragroup stability.
6. Train baseline classifiers without subject leakage.
7. Compare lateralized results against the regional_v1 baseline.

The notebook uses the public OpenNeuro/NEMAR preprocessed derivatives and
does not operationalize device-specific Contec or Emotiv models.

## 1. Setup

The notebook is designed to run from the `notebooks/` directory. It writes
outputs to `results/` and figures to `figures/`.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=RuntimeWarning)

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / "src"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

for path in [SRC_DIR, FIGURES_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import information_theory_measures as itm
import itm_algorithmic_complexity as itm_ac
import itm_geometry as geom

from matplotlib.ticker import FormatStrFormatter

RANDOM_STATE = 17536
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_theme(style="whitegrid", font="Arial")

PROJECT_ROOT

## 2. Parameters

The full lateralized extraction can be expensive because it recomputes
HCFIM features from the EEG time series. Keep the pilot enabled first.
After the smoke run is correct, set `RUN_FULL_LATERALIZED_FEATURE_EXTRACTION = True`.

In [ ]:
def resolve_dataset_root(project_root: Path) -> Path:
    """Locate ds004504 derivatives or use HCFIM_DATASET_ROOT."""
    candidates = []
    env_root = os.environ.get("HCFIM_DATASET_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        project_root / "data" / "ds004504",
        project_root / "data" / "data-08-00095",
        project_root / "data",
    ])
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "participants.tsv").is_file() and (candidate / "derivatives").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate ds004504. Set HCFIM_DATASET_ROOT to the dataset directory "
        "containing participants.tsv and derivatives/."
    )

DATASET_ROOT = resolve_dataset_root(PROJECT_ROOT)
DERIVATIVES_ROOT = DATASET_ROOT / "derivatives"
PARTICIPANTS_TSV = DATASET_ROOT / "participants.tsv"

RUN_LATERALIZED_PILOT = False
RUN_FULL_LATERALIZED_FEATURE_EXTRACTION = False
RUN_LATERALIZED_CLASSIFICATION = False
RUN_DERIVED_APPENDIX_FEATURES = False
FORCE_RECOMPUTE_LATERALIZED_PARTS = False
WRITE_COMBINED_LATERALIZED_EPOCH_CSV = False

PILOT_SUBJECTS_PER_GROUP = 2
MAX_EPOCHS_PER_SUBJECT_PILOT = 4
MAX_EPOCHS_PER_SUBJECT_FULL = None

EPOCH_LENGTH_S = 10.0
EPOCH_OVERLAP = 0.50

CORE_BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "broadband": (0.5, 40.0),
}

LATERALIZED_V1 = {
    "frontal_left": ["Fp1", "F7", "F3"],
    "frontal_right": ["Fp2", "F8", "F4"],
    "temporal_left": ["T7", "P7"],
    "temporal_right": ["T8", "P8"],
    "parietal_left": ["P3"],
    "parietal_right": ["P4"],
    "occipital_left": ["O1"],
    "occipital_right": ["O2"],
    "central_midline_extended": ["C3", "Cz", "C4", "Fz", "Pz"],
}

LATERAL_PAIRS = {
    "frontal": ("frontal_left", "frontal_right"),
    "temporal": ("temporal_left", "temporal_right"),
    "parietal": ("parietal_left", "parietal_right"),
    "occipital": ("occipital_left", "occipital_right"),
}

CHANNEL_RENAME = {
    "T3": "T7",
    "T4": "T8",
    "T5": "P7",
    "T6": "P8",
}

GROUP_MAP = {
    "A": "alzheimer",
    "F": "frontotemporal_dementia",
    "C": "control",
}

CLASSIFICATION_COMPARISONS = {
    "control_vs_alzheimer": ["control", "alzheimer"],
    "control_vs_frontotemporal_dementia": ["control", "frontotemporal_dementia"],
}

PARAMS = {
    "analysis_schema": "lateralized_v1",
    "dataset_root": str(DATASET_ROOT),
    "derivatives_root": str(DERIVATIVES_ROOT),
    "epoch_length_s": EPOCH_LENGTH_S,
    "epoch_overlap": EPOCH_OVERLAP,
    "bands": CORE_BANDS,
    "regions": LATERALIZED_V1,
    "lateral_pairs": LATERAL_PAIRS,
    "random_state": RANDOM_STATE,
}

with open(RESULTS_DIR / "analysis_parameters_lateralized_v1.json", "w", encoding="utf-8") as f:
    json.dump(PARAMS, f, indent=2)

PARAMS

## 3. Dataset Index

Participants are indexed from the public OpenNeuro dataset metadata. The
class labels are mapped as: `C = control`, `A = Alzheimer`, `F =
frontotemporal dementia`.

In [ ]:
def load_participants(participants_tsv: Path) -> pd.DataFrame:
    participants = pd.read_csv(participants_tsv, sep="\t")
    participants = participants.rename(columns={
        "Gender": "gender",
        "Age": "age",
        "Group": "group_code",
    })
    participants["diagnosis"] = participants["group_code"].map(GROUP_MAP)
    participants["set_path"] = participants["participant_id"].apply(
        lambda sid: DERIVATIVES_ROOT / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    )
    participants["set_exists"] = participants["set_path"].apply(lambda p: Path(p).exists())
    return participants


participants = load_participants(PARTICIPANTS_TSV)
participants_summary = (
    participants
    .groupby("diagnosis", dropna=False)
    .agg(
        n=("participant_id", "count"),
        mean_age=("age", "mean"),
        sd_age=("age", "std"),
        mean_mmse=("MMSE", "mean"),
        sd_mmse=("MMSE", "std"),
        available_sets=("set_exists", "sum"),
    )
    .reset_index()
)

participants.to_csv(RESULTS_DIR / "participants_index_lateralized_v1.csv", index=False)
participants_summary

## 4. Feature Extraction Functions

HCFIM features are computed from finite probability vectors induced by
symbolic transforms. The main transforms are ordinal Shannon, Renyi and
Tsallis variants. Derived/algorithmic transforms can be enabled for an
appendix run.

In [ ]:
def require_mne():
    try:
        import mne
    except Exception as exc:
        raise ImportError(
            "MNE is required to read EEGLAB .set files. Install with `pip install mne`."
        ) from exc
    return mne


def canonical_channel_name(name: str) -> str:
    base = str(name).strip()
    return CHANNEL_RENAME.get(base, base)


def load_eeglab_subject(set_path: Path):
    mne = require_mne()
    raw = mne.io.read_raw_eeglab(str(set_path), preload=True, verbose="ERROR")
    rename_map = {ch: canonical_channel_name(ch) for ch in raw.ch_names if canonical_channel_name(ch) != ch}
    if rename_map:
        raw.rename_channels(rename_map)
    raw.pick_types(eeg=True)
    data = raw.get_data()
    channels = list(raw.ch_names)
    sfreq = float(raw.info["sfreq"])
    return data, sfreq, channels


def epoch_slices(n_samples: int, sfreq: float, epoch_length_s: float, overlap: float):
    window = int(round(epoch_length_s * sfreq))
    step = int(round(window * (1.0 - overlap)))
    if window <= 0 or step <= 0:
        raise ValueError("Invalid epoching parameters.")
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        yield epoch_id, start, start + window


def bandpass_1d(x: np.ndarray, sfreq: float, band: tuple[float, float]) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = signal.detrend(x, type="constant")
    low, high = band
    nyq = sfreq / 2.0
    low = max(float(low), 0.01)
    high = min(float(high), nyq - 0.5)
    if high <= low:
        return x
    sos = signal.butter(4, [low / nyq, high / nyq], btype="bandpass", output="sos")
    return signal.sosfiltfilt(sos, x)


def integrate_power(y: np.ndarray, x: np.ndarray) -> float:
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))


def region_signal(window: np.ndarray, channels: list[str], region_channels: list[str]) -> np.ndarray | None:
    channel_index = {canonical_channel_name(ch): idx for idx, ch in enumerate(channels)}
    indices = [channel_index[ch] for ch in region_channels if ch in channel_index]
    if not indices:
        return None
    return np.nanmean(window[indices, :], axis=0)


def relative_bandpower_features(x: np.ndarray, sfreq: float, region_name: str) -> list[dict]:
    nperseg = min(int(round(sfreq * 2.0)), x.size)
    if nperseg < 8:
        return []
    freqs, psd = signal.welch(x, fs=sfreq, nperseg=nperseg, detrend="constant")
    total_mask = (freqs >= 0.5) & (freqs < 40.0)
    total_power = integrate_power(psd[total_mask], freqs[total_mask]) if np.any(total_mask) else np.nan
    if not np.isfinite(total_power) or total_power <= 0:
        return []

    rows = []
    band_values = {}
    for band_name, band in CORE_BANDS.items():
        if band_name == "broadband":
            continue
        mask = (freqs >= band[0]) & (freqs < band[1])
        power = integrate_power(psd[mask], freqs[mask]) if np.any(mask) else np.nan
        band_values[band_name] = float(power / total_power) if np.isfinite(power) else np.nan
        rows.append({
            "feature_family": "psd",
            "feature_name": f"psd__{region_name}__relative_bandpower_{band_name}",
            "value": band_values[band_name],
        })

    alpha = band_values.get("alpha", np.nan)
    beta = band_values.get("beta", np.nan)
    theta = band_values.get("theta", np.nan)
    rows.extend([
        {
            "feature_family": "psd",
            "feature_name": f"psd__{region_name}__theta_alpha_ratio",
            "value": float(theta / alpha) if alpha and alpha > 0 else np.nan,
        },
        {
            "feature_family": "psd",
            "feature_name": f"psd__{region_name}__alpha_beta_ratio",
            "value": float(alpha / beta) if beta and beta > 0 else np.nan,
        },
    ])

    psd_total = psd[total_mask]
    probs = psd_total / np.sum(psd_total)
    probs = np.clip(probs, np.finfo(float).tiny, 1.0)
    spectral_entropy = -np.sum(probs * np.log2(probs)) / np.log2(probs.size)
    rows.append({
        "feature_family": "psd",
        "feature_name": f"psd__{region_name}__spectral_entropy",
        "value": float(spectral_entropy),
    })
    return rows


TRANSFORMS_MAIN = [
    {"transform_id": "ordinal_shannon_m3_t1", "kind": "ordinal", "entropy_kind": "shannon", "m": 3, "tau": 1},
    {"transform_id": "ordinal_shannon_m4_t1", "kind": "ordinal", "entropy_kind": "shannon", "m": 4, "tau": 1},
    {"transform_id": "ordinal_shannon_m5_t1", "kind": "ordinal", "entropy_kind": "shannon", "m": 5, "tau": 1},
    {"transform_id": "ordinal_renyi_a08_m3_t1", "kind": "ordinal", "entropy_kind": "renyi", "alpha": 0.8, "m": 3, "tau": 1},
    {"transform_id": "ordinal_renyi_a12_m3_t1", "kind": "ordinal", "entropy_kind": "renyi", "alpha": 1.2, "m": 3, "tau": 1},
    {"transform_id": "ordinal_tsallis_q08_m3_t1", "kind": "ordinal", "entropy_kind": "tsallis", "q": 0.8, "m": 3, "tau": 1},
    {"transform_id": "ordinal_tsallis_q12_m3_t1", "kind": "ordinal", "entropy_kind": "tsallis", "q": 1.2, "m": 3, "tau": 1},
]

TRANSFORMS_DERIVED_APPENDIX = [
    {"transform_id": "apen_distance_m2_t1_b16", "kind": "derived", "derived_id": "apen_distance_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "sampen_distance_m2_t1_b16", "kind": "derived", "derived_id": "sampen_distance_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "fuzzy_similarity_m2_t1_b16", "kind": "derived", "derived_id": "fuzzy_similarity_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "k2_correlation_m2_t1_b16", "kind": "derived", "derived_id": "k2_correlation_pdf", "m": 2, "tau": 1, "bins": 16},
    {"transform_id": "lz_ordinal_phrase_m3_t1_b16", "kind": "derived", "derived_id": "lz_ordinal_phrase_pdf", "m": 3, "tau": 1, "bins": 16},
]


def entropy_spec_for_transform(transform: dict, n_states: int) -> geom.EntropySpec:
    return geom.EntropySpec(
        entropy_id=transform["transform_id"],
        label=transform["transform_id"],
        family=transform["kind"],
        probability_domain=transform.get("derived_id", "Bandt-Pompe ordinal patterns"),
        n_states=int(n_states),
        entropy_kind=transform.get("entropy_kind", "shannon"),
        m=transform.get("m"),
        tau=transform.get("tau"),
        q=transform.get("q"),
        alpha=transform.get("alpha"),
    )


def probability_for_transform(x: np.ndarray, transform: dict):
    if transform["kind"] == "ordinal":
        m = int(transform.get("m", 3))
        tau = int(transform.get("tau", 1))
        probabilities = geom.ordinal_probabilities(x, m=m, tau=tau)
        return probabilities, entropy_spec_for_transform(transform, n_states=math.factorial(m))

    if transform["kind"] == "derived":
        probabilities = itm_ac.derived_probability_vector(
            x,
            derived_id=transform["derived_id"],
            m=int(transform.get("m", 2)),
            tau=int(transform.get("tau", 1)),
            bins=int(transform.get("bins", 16)),
        )
        return probabilities, entropy_spec_for_transform(transform, n_states=len(probabilities))

    raise ValueError(f"Unknown transform kind: {transform['kind']}")


def hcfim_feature_rows(x: np.ndarray, region_name: str, band_name: str, transforms: list[dict]) -> list[dict]:
    rows = []
    for transform in transforms:
        try:
            probabilities, spec = probability_for_transform(x, transform)
            for distance_name, distance_meta in geom.DISTANCES.items():
                point = geom.hcfim_point(probabilities, spec, distance_code=distance_meta["code"])
                for coord_name in ["H", "C", "FIM"]:
                    rows.append({
                        "feature_family": "hcfim",
                        "feature_name": (
                            f"hcfim__{transform['transform_id']}__{distance_name}"
                            f"__{region_name}__{band_name}__{coord_name}"
                        ),
                        "value": float(point[coord_name]),
                    })
        except Exception as exc:
            rows.append({
                "feature_family": "hcfim_error",
                "feature_name": f"hcfim_error__{transform['transform_id']}__{region_name}__{band_name}",
                "value": np.nan,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return rows

## 5. Extract Lateralized Epoch Features

Pilot outputs are intentionally small. Full outputs are written with a
`lateralized_v1` suffix so they do not overwrite the regional_v1 paper
results. The full extraction writes one CSV per subject in a `_parts`
folder and can be resumed without recomputing completed subjects.

In [ ]:
def select_balanced_pilot(participants: pd.DataFrame, n_per_group: int) -> pd.DataFrame:
    return (
        participants[participants["set_exists"]]
        .sort_values(["diagnosis", "participant_id"])
        .groupby("diagnosis", group_keys=False)
        .head(n_per_group)
        .reset_index(drop=True)
    )


def extract_epoch_features_for_subject(
    subject_row,
    regions: dict[str, list[str]],
    transforms: list[dict],
    max_epochs: int | None = None,
) -> pd.DataFrame:
    data, sfreq, channels = load_eeglab_subject(Path(subject_row.set_path))
    rows = []
    for epoch_id, start, end in epoch_slices(data.shape[1], sfreq, EPOCH_LENGTH_S, EPOCH_OVERLAP):
        if max_epochs is not None and epoch_id >= int(max_epochs):
            break
        window = data[:, start:end]
        base = {
            "participant_id": subject_row.participant_id,
            "diagnosis": subject_row.diagnosis,
            "group_code": subject_row.group_code,
            "epoch_id": epoch_id,
            "start_sample": start,
            "end_sample": end,
            "sfreq": sfreq,
        }
        for region_name, region_channels in regions.items():
            x_region = region_signal(window, channels, region_channels)
            if x_region is None:
                continue

            for psd_row in relative_bandpower_features(x_region, sfreq, region_name):
                rows.append({**base, "region": region_name, "band": "broadband", **psd_row})

            for band_name, band in CORE_BANDS.items():
                x_band = bandpass_1d(x_region, sfreq, band)
                for feature_row in hcfim_feature_rows(x_band, region_name, band_name, transforms):
                    rows.append({**base, "region": region_name, "band": band_name, **feature_row})
    return pd.DataFrame(rows)


def extract_feature_table(
    selected_subjects: pd.DataFrame,
    regions: dict[str, list[str]],
    transforms: list[dict],
    max_epochs_per_subject: int | None,
    output_name: str,
    *,
    force_recompute: bool = False,
    write_combined_csv: bool = False,
    return_table: bool = True,
) -> tuple[pd.DataFrame, Path]:
    parts_dir = RESULTS_DIR / f"{output_name}_parts"
    parts_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []

    for idx, subject_row in enumerate(selected_subjects.itertuples(index=False), start=1):
        part_path = parts_dir / f"{subject_row.participant_id}.csv"
        if part_path.exists() and not force_recompute:
            print(f"[{idx}/{len(selected_subjects)}] Reusing {subject_row.participant_id} -> {part_path.name}")
            manifest_rows.append(
                {
                    "participant_id": subject_row.participant_id,
                    "diagnosis": subject_row.diagnosis,
                    "part_path": str(part_path),
                    "status": "reused",
                    "rows": np.nan,
                }
            )
            continue

        print(f"[{idx}/{len(selected_subjects)}] Extracting {subject_row.participant_id} ({subject_row.diagnosis})")
        subject_features = extract_epoch_features_for_subject(
            subject_row,
            regions=regions,
            transforms=transforms,
            max_epochs=max_epochs_per_subject,
        )
        subject_features.to_csv(part_path, index=False)
        manifest_rows.append(
            {
                "participant_id": subject_row.participant_id,
                "diagnosis": subject_row.diagnosis,
                "part_path": str(part_path),
                "status": "computed",
                "rows": int(len(subject_features)),
            }
        )
        print(f"Saved {part_path} ({len(subject_features):,} rows)")

    manifest = pd.DataFrame(manifest_rows)
    manifest_path = RESULTS_DIR / f"{output_name}_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    print(f"Saved manifest {manifest_path}")

    part_files = sorted(parts_dir.glob("*.csv"))
    combined_path = RESULTS_DIR / f"{output_name}.csv"
    if write_combined_csv and part_files:
        with combined_path.open("w", encoding="utf-8", newline="") as output_file:
            for part_idx, part_file in enumerate(part_files):
                for chunk_idx, chunk in enumerate(pd.read_csv(part_file, chunksize=200_000)):
                    chunk.to_csv(
                        output_file,
                        index=False,
                        header=(part_idx == 0 and chunk_idx == 0),
                    )
        print(f"Saved combined CSV {combined_path}")

    if return_table and part_files:
        return pd.concat((pd.read_csv(part_file) for part_file in part_files), ignore_index=True), parts_dir
    return pd.DataFrame(), parts_dir


active_transforms = TRANSFORMS_MAIN + (TRANSFORMS_DERIVED_APPENDIX if RUN_DERIVED_APPENDIX_FEATURES else [])

if RUN_LATERALIZED_PILOT:
    pilot_subjects = select_balanced_pilot(participants, PILOT_SUBJECTS_PER_GROUP)
    epoch_features_lateralized_v1_pilot, epoch_features_lateralized_v1_pilot_parts_dir = extract_feature_table(
        pilot_subjects,
        regions=LATERALIZED_V1,
        transforms=active_transforms,
        max_epochs_per_subject=MAX_EPOCHS_PER_SUBJECT_PILOT,
        output_name="epoch_features_lateralized_v1_pilot",
        force_recompute=FORCE_RECOMPUTE_LATERALIZED_PARTS,
        write_combined_csv=True,
        return_table=True,
    )
else:
    pilot_path = RESULTS_DIR / "epoch_features_lateralized_v1_pilot.csv"
    epoch_features_lateralized_v1_pilot = pd.read_csv(pilot_path) if pilot_path.exists() else pd.DataFrame()
    epoch_features_lateralized_v1_pilot_parts_dir = RESULTS_DIR / "epoch_features_lateralized_v1_pilot_parts"

if RUN_FULL_LATERALIZED_FEATURE_EXTRACTION:
    full_subjects = participants[participants["set_exists"]].reset_index(drop=True)
    epoch_features_lateralized_v1_full, epoch_features_lateralized_v1_full_parts_dir = extract_feature_table(
        full_subjects,
        regions=LATERALIZED_V1,
        transforms=active_transforms,
        max_epochs_per_subject=MAX_EPOCHS_PER_SUBJECT_FULL,
        output_name="epoch_features_lateralized_v1_full",
        force_recompute=FORCE_RECOMPUTE_LATERALIZED_PARTS,
        write_combined_csv=WRITE_COMBINED_LATERALIZED_EPOCH_CSV,
        return_table=False,
    )
else:
    full_path = RESULTS_DIR / "epoch_features_lateralized_v1_full.csv"
    epoch_features_lateralized_v1_full = pd.DataFrame()
    epoch_features_lateralized_v1_full_parts_dir = RESULTS_DIR / "epoch_features_lateralized_v1_full_parts"
    if full_path.exists() and not epoch_features_lateralized_v1_full_parts_dir.exists():
        print(f"Combined full CSV exists at {full_path}; aggregation from the combined CSV is not loaded by default to avoid high memory use.")

epoch_features_lateralized_v1_pilot.head() if not epoch_features_lateralized_v1_pilot.empty else "Pilot extraction not run."

## 6. Aggregate To Subject Matrix And Add Lateralization Deltas

The classifier matrix is one row per subject. This prevents epoch-level
leakage. Deltas are computed as `right - left` for homologous regions.

In [ ]:
def aggregate_subject_features(epoch_features: pd.DataFrame, output_name: str) -> pd.DataFrame:
    if epoch_features.empty:
        return pd.DataFrame()

    aggregated = summarize_epoch_features(epoch_features)
    return aggregated_features_to_matrix(aggregated, output_name)


def summarize_epoch_features(epoch_features: pd.DataFrame) -> pd.DataFrame:
    def iqr(values):
        return np.nanpercentile(values, 75) - np.nanpercentile(values, 25)

    if epoch_features.empty:
        return pd.DataFrame()

    clean = epoch_features[
        epoch_features["feature_family"].isin(["psd", "hcfim"])
    ].copy()
    clean["value"] = pd.to_numeric(clean["value"], errors="coerce")
    clean = clean[np.isfinite(clean["value"])]
    if clean.empty:
        return pd.DataFrame()

    return (
        clean
        .groupby(["participant_id", "diagnosis", "feature_family", "feature_name"])["value"]
        .agg(
            mean="mean",
            std="std",
            median="median",
            iqr=iqr,
            p25=lambda x: np.nanpercentile(x, 25),
            p75=lambda x: np.nanpercentile(x, 75),
        )
        .reset_index()
    )


def aggregated_features_to_matrix(aggregated: pd.DataFrame, output_name: str) -> pd.DataFrame:
    if aggregated.empty:
        return pd.DataFrame()

    long_rows = []
    for row in aggregated.itertuples(index=False):
        for agg_name in ["mean", "std", "median", "iqr", "p25", "p75"]:
            long_rows.append({
                "participant_id": row.participant_id,
                "diagnosis": row.diagnosis,
                "feature_family": row.feature_family,
                "feature_name": f"{row.feature_name}__{agg_name}",
                "value": getattr(row, agg_name),
            })

    long_features = pd.DataFrame(long_rows)
    matrix = (
        long_features
        .pivot_table(index=["participant_id", "diagnosis"], columns="feature_name", values="value", aggfunc="first")
        .reset_index()
    )
    matrix.columns.name = None
    matrix.to_csv(RESULTS_DIR / f"{output_name}.csv", index=False)
    return matrix


def aggregate_subject_features_from_parts(parts_dir: Path, output_name: str) -> pd.DataFrame:
    part_files = sorted(Path(parts_dir).glob("*.csv"))
    if not part_files:
        return pd.DataFrame()

    summaries = []
    usecols = ["participant_id", "diagnosis", "feature_family", "feature_name", "value"]
    for idx, part_file in enumerate(part_files, start=1):
        print(f"[{idx}/{len(part_files)}] Aggregating {part_file.name}")
        part = pd.read_csv(part_file, usecols=usecols)
        summary = summarize_epoch_features(part)
        if not summary.empty:
            summaries.append(summary)

    if not summaries:
        return pd.DataFrame()
    aggregated = pd.concat(summaries, ignore_index=True)
    return aggregated_features_to_matrix(aggregated, output_name)


def load_subject_matrix(output_name: str) -> pd.DataFrame:
    path = RESULTS_DIR / f"{output_name}.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def parts_completion_status(parts_dir: Path, expected_subject_ids: list[str]) -> tuple[bool, list[str]]:
    existing = {part_file.stem for part_file in Path(parts_dir).glob("*.csv")}
    expected = [str(subject_id) for subject_id in expected_subject_ids]
    missing = [subject_id for subject_id in expected if subject_id not in existing]
    print(f"Parts status for {Path(parts_dir).name}: {len(existing)}/{len(expected)} subjects available.")
    if missing:
        print(f"Missing subjects: {missing[:10]}{' ...' if len(missing) > 10 else ''}")
    return len(missing) == 0, missing


def add_lateralization_delta_features(matrix: pd.DataFrame, output_name: str | None = None) -> pd.DataFrame:
    if matrix.empty:
        return matrix
    out = matrix.copy()
    source_cols = [c for c in out.columns if c.startswith("psd__") or c.startswith("hcfim__")]
    for base_region, (left_region, right_region) in LATERAL_PAIRS.items():
        left_token = f"__{left_region}__"
        right_token = f"__{right_region}__"
        for left_col in source_cols:
            if left_token not in left_col:
                continue
            right_col = left_col.replace(left_token, right_token)
            if right_col not in out.columns:
                continue
            delta_col = "latdiff__" + left_col.replace(left_token, f"__{base_region}__")
            out[delta_col] = out[right_col] - out[left_col]
    if output_name:
        out.to_csv(RESULTS_DIR / f"{output_name}.csv", index=False)
    return out


subject_features_lateralized_v1_pilot = aggregate_subject_features(
    epoch_features_lateralized_v1_pilot,
    "subject_features_lateralized_v1_pilot_no_deltas",
)
subject_features_lateralized_v1_pilot = add_lateralization_delta_features(
    subject_features_lateralized_v1_pilot,
    "subject_features_lateralized_v1_pilot",
)

full_expected_subject_ids = participants.loc[participants["set_exists"], "participant_id"].astype(str).tolist()
full_parts_complete, missing_full_parts = parts_completion_status(
    epoch_features_lateralized_v1_full_parts_dir,
    full_expected_subject_ids,
)

if Path(epoch_features_lateralized_v1_full_parts_dir).exists() and full_parts_complete:
    subject_features_lateralized_v1_full = aggregate_subject_features_from_parts(
        epoch_features_lateralized_v1_full_parts_dir,
        "subject_features_lateralized_v1_full_no_deltas",
    )
elif Path(epoch_features_lateralized_v1_full_parts_dir).exists() and not full_parts_complete:
    print("Full lateralized parts are incomplete. Skipping full subject matrix and classifiers until all subjects are available.")
    subject_features_lateralized_v1_full = pd.DataFrame()
elif not epoch_features_lateralized_v1_full.empty:
    subject_features_lateralized_v1_full = aggregate_subject_features(
        epoch_features_lateralized_v1_full,
        "subject_features_lateralized_v1_full_no_deltas",
    )
else:
    subject_features_lateralized_v1_full = load_subject_matrix("subject_features_lateralized_v1_full_no_deltas")

subject_features_lateralized_v1_full = add_lateralization_delta_features(
    subject_features_lateralized_v1_full,
    "subject_features_lateralized_v1_full",
)

subject_features_lateralized_v1 = (
    subject_features_lateralized_v1_full
    if not subject_features_lateralized_v1_full.empty
    else subject_features_lateralized_v1_pilot
)

feature_summary = {
    "n_subjects": int(subject_features_lateralized_v1.shape[0]),
    "n_psd_features": int(sum(c.startswith("psd__") for c in subject_features_lateralized_v1.columns)),
    "n_hcfim_features": int(sum(c.startswith("hcfim__") for c in subject_features_lateralized_v1.columns)),
    "n_lateral_delta_features": int(sum(c.startswith("latdiff__") for c in subject_features_lateralized_v1.columns)),
    "full_parts_complete": bool(full_parts_complete),
    "n_missing_full_subject_parts": int(len(missing_full_parts)),
}
feature_summary

## 7. HCFIM Geometric Ranking

Each candidate HCFIM geometry is ranked by centroid distance, within-group
spread, inter/intra ratio and silhouette score. This section is descriptive
and does not perform supervised feature selection.

In [ ]:
def parse_hcfim_feature_name(feature_name: str) -> dict:
    # hcfim__transform__distance__region__band__coord__agg
    parts = feature_name.split("__")
    if len(parts) < 7 or parts[0] != "hcfim":
        return {}
    return {
        "transform_id": parts[1],
        "distance": parts[2],
        "region": parts[3],
        "band": parts[4],
        "coord": parts[5],
        "agg": parts[6],
    }


def rank_hcfim_geometries(subject_features: pd.DataFrame, comparison_name: str, groups: list[str]) -> pd.DataFrame:
    if subject_features.empty:
        return pd.DataFrame()
    subset = subject_features[subject_features["diagnosis"].isin(groups)].copy()
    if subset["diagnosis"].nunique() != 2:
        return pd.DataFrame()

    hcfim_cols = [c for c in subset.columns if c.startswith("hcfim__") and c.endswith("__mean")]
    parsed = pd.DataFrame([{**parse_hcfim_feature_name(c), "column": c} for c in hcfim_cols])
    parsed = parsed.dropna(subset=["transform_id", "distance", "region", "band", "coord"], how="any")

    rows = []
    for keys, group_cols in parsed.groupby(["transform_id", "distance", "region", "band"]):
        coord_cols = {}
        for coord in ["H", "C", "FIM"]:
            match = group_cols[group_cols["coord"] == coord]
            if not match.empty:
                coord_cols[coord] = match.iloc[0]["column"]
        if set(coord_cols) != {"H", "C", "FIM"}:
            continue

        data = subset[["participant_id", "diagnosis", *coord_cols.values()]].dropna()
        if data["diagnosis"].nunique() != 2 or len(data) < 6:
            continue

        control_group, pathology_group = groups
        control = data[data["diagnosis"] == control_group][list(coord_cols.values())].to_numpy(float)
        pathology = data[data["diagnosis"] == pathology_group][list(coord_cols.values())].to_numpy(float)
        if len(control) < 2 or len(pathology) < 2:
            continue

        centroid_control = control.mean(axis=0)
        centroid_pathology = pathology.mean(axis=0)
        centroid_distance = float(np.linalg.norm(centroid_pathology - centroid_control))
        within_control = float(np.mean(np.linalg.norm(control - centroid_control, axis=1)))
        within_pathology = float(np.mean(np.linalg.norm(pathology - centroid_pathology, axis=1)))
        within_spread = np.nanmean([within_control, within_pathology])
        inter_intra_ratio = centroid_distance / within_spread if within_spread > 0 else np.nan

        labels = (data["diagnosis"] == pathology_group).astype(int).to_numpy()
        try:
            silhouette = silhouette_score(data[list(coord_cols.values())].to_numpy(float), labels)
        except Exception:
            silhouette = np.nan

        transform_id, distance, region, band = keys
        rows.append({
            "schema": "lateralized_v1",
            "comparison": comparison_name,
            "control_group": control_group,
            "pathology_group": pathology_group,
            "transform_id": transform_id,
            "distance": distance,
            "region": region,
            "band": band,
            "n_subjects": len(data),
            "centroid_distance": centroid_distance,
            "within_spread": within_spread,
            "inter_intra_ratio": inter_intra_ratio,
            "silhouette": silhouette,
            "delta_H": float(centroid_pathology[0] - centroid_control[0]),
            "delta_C": float(centroid_pathology[1] - centroid_control[1]),
            "delta_FIM": float(centroid_pathology[2] - centroid_control[2]),
        })
    return pd.DataFrame(rows).sort_values(["inter_intra_ratio", "centroid_distance"], ascending=False)


all_rankings = []
for comparison_name, groups in CLASSIFICATION_COMPARISONS.items():
    ranking = rank_hcfim_geometries(subject_features_lateralized_v1, comparison_name, groups)
    if not ranking.empty:
        ranking.to_csv(RESULTS_DIR / f"hcfim_geometric_ranking_lateralized_v1_{comparison_name}.csv", index=False)
        all_rankings.append(ranking)

hcfim_rankings_lateralized_v1 = pd.concat(all_rankings, ignore_index=True) if all_rankings else pd.DataFrame()
hcfim_rankings_lateralized_v1.head(20) if not hcfim_rankings_lateralized_v1.empty else "No lateralized geometric ranking available yet."

## 8. Plot Top Lateralized HCFIM Spaces

The plots show the top-ranked HCFIM space per binary comparison. These
are exploratory visual summaries, not decision boundaries.

In [ ]:
def plot_top_hcfim_space(subject_features: pd.DataFrame, ranking_row: pd.Series, output_path: Path):
    transform_id = ranking_row["transform_id"]
    distance = ranking_row["distance"]
    region = ranking_row["region"]
    band = ranking_row["band"]
    groups = [ranking_row["control_group"], ranking_row["pathology_group"]]
    cols = {
        coord: f"hcfim__{transform_id}__{distance}__{region}__{band}__{coord}__mean"
        for coord in ["H", "C", "FIM"]
    }
    data = subject_features[subject_features["diagnosis"].isin(groups)][["participant_id", "diagnosis", *cols.values()]].dropna()
    data = data.rename(columns={v: k for k, v in cols.items()})
    if data.empty:
        return None

    colors = {"control": "#4C78A8", "alzheimer": "#E45756", "frontotemporal_dementia": "#72B7B2"}
    colorsC = {"control": "#1e5a9b", "alzheimer": "#972726", "frontotemporal_dementia": "#158078"}
    fig = plt.figure(figsize=(9, 7))

    plt.rcParams.update({'font.size': 12})
    plt.rcParams.update({
        'axes.titlesize': 10,     # Title size
        'axes.labelsize': 11,     # X and Y label size
        'xtick.labelsize': 9,    # X-tick size
        'ytick.labelsize': 9,    # Y-tick size
        'legend.fontsize': 9,    # Legend size
        'figure.titlesize': 12    # Super title size
    })
    
    ax = fig.add_subplot(111, projection="3d")
    for diagnosis, group in data.groupby("diagnosis"):
        ax.scatter(group["H"], group["C"], group["FIM"], s=55, alpha=0.70,
                   c=colors.get(diagnosis, "#666666"), label=diagnosis)
        centroid = group[["H", "C", "FIM"]].mean()
        ax.scatter(
            centroid["H"], centroid["C"], centroid["FIM"],
            marker="X", s=190, c=colorsC.get(diagnosis, "#666666"),
            edgecolors="white", linewidths=1.2, label=f"centroid {diagnosis}",
        )
    ax.set_xlabel("H")
    ax.set_ylabel("C")
    ax.set_zlabel("FIM")

    ax.xaxis.pane.set_facecolor((1, 1, 1, 1))
    ax.yaxis.pane.set_facecolor((1, 1, 1, 1))
    ax.zaxis.pane.set_facecolor((1, 1, 1, 1))

    ax.xaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    ax.zaxis.set_major_formatter(FormatStrFormatter("%.4f"))
    
    ax.view_init(elev=24, azim=-58)
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return output_path


hcfim_lateralized_figure_paths = []
if not hcfim_rankings_lateralized_v1.empty:
    for comparison_name, ranking in hcfim_rankings_lateralized_v1.groupby("comparison"):
        top = ranking.iloc[0]
        path = FIGURES_DIR / f"hcfim_space_top_lateralized_v1_{comparison_name}.png"
        hcfim_lateralized_figure_paths.append(plot_top_hcfim_space(subject_features_lateralized_v1, top, path))
else:
    print("Run feature extraction and geometric ranking before plotting lateralized HCFIM spaces.")

hcfim_lateralized_figure_paths

## 9. Classifiers Without Subject Leakage

The classifier input is the subject-level matrix. Scaling and feature
selection are inside the cross-validation pipeline.

In [ ]:
def feature_columns_for_family(df: pd.DataFrame, feature_family: str) -> list[str]:
    if feature_family == "psd":
        return [c for c in df.columns if c.startswith("psd__")]
    if feature_family == "hcfim":
        return [c for c in df.columns if c.startswith("hcfim__")]
    if feature_family == "psd_plus_hcfim":
        return [c for c in df.columns if c.startswith("psd__") or c.startswith("hcfim__")]
    if feature_family == "lateral_deltas":
        return [c for c in df.columns if c.startswith("latdiff__")]
    if feature_family == "psd_plus_hcfim_plus_lateral_deltas":
        return [c for c in df.columns if c.startswith("psd__") or c.startswith("hcfim__") or c.startswith("latdiff__")]
    raise ValueError(f"Unknown feature family: {feature_family}")


def classifier_grid():
    return {
        "logistic_elasticnet": (
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            {
                "model__C": [0.05, 0.1, 0.5, 1.0],
                "model__l1_ratio": [0.15, 0.5, 0.85],
            },
        ),
        "svm_rbf": (
            SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
            {
                "model__C": [0.1, 1.0, 10.0],
                "model__gamma": ["scale", 0.01, 0.1],
            },
        ),
        "random_forest": (
            RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
            {
                "model__n_estimators": [200],
                "model__max_depth": [2, 3, None],
            },
        ),
        "extra_trees": (
            ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE),
            {
                "model__n_estimators": [300],
                "model__max_depth": [2, 3, None],
            },
        ),
        "gradient_boosting": (
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            {
                "model__n_estimators": [50, 100],
                "model__learning_rate": [0.03, 0.1],
                "model__max_depth": [1, 2],
            },
        ),
        "mlp_small": (
            MLPClassifier(
                hidden_layer_sizes=(8,),
                alpha=0.01,
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
            {
                "model__alpha": [0.001, 0.01, 0.1],
                "model__hidden_layer_sizes": [(8,), (12,)],
            },
        ),
    }


def evaluate_feature_family(df: pd.DataFrame, comparison_name: str, groups: list[str], feature_family: str) -> pd.DataFrame:
    subset = df[df["diagnosis"].isin(groups)].copy()
    feature_cols = feature_columns_for_family(subset, feature_family)
    if len(feature_cols) == 0 or subset["diagnosis"].nunique() != 2:
        return pd.DataFrame()

    X = subset[feature_cols].to_numpy(float)
    y = (subset["diagnosis"] == groups[1]).astype(int).to_numpy()

    outer_cv = StratifiedKFold(n_splits=min(5, np.bincount(y).min()), shuffle=True, random_state=RANDOM_STATE)
    model_defs = classifier_grid()
    rows = []

    for model_name, (model, grid) in model_defs.items():
        oof_prob = np.full(len(y), np.nan)
        oof_pred = np.full(len(y), np.nan)
        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
            k = min(30, max(3, X.shape[1] // 10), X.shape[1])
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("select", SelectKBest(score_func=f_classif, k=k)),
                ("model", model),
            ])
            inner_cv = StratifiedKFold(
                n_splits=min(3, np.bincount(y[train_idx]).min()),
                shuffle=True,
                random_state=RANDOM_STATE + fold,
            )
            search = GridSearchCV(
                pipe,
                param_grid=grid,
                scoring="roc_auc",
                cv=inner_cv,
                n_jobs=-1,
                error_score=np.nan,
            )
            search.fit(X[train_idx], y[train_idx])
            best = search.best_estimator_
            if hasattr(best, "predict_proba"):
                prob = best.predict_proba(X[test_idx])[:, 1]
            else:
                scores = best.decision_function(X[test_idx])
                prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
            pred = (prob >= 0.5).astype(int)
            oof_prob[test_idx] = prob
            oof_pred[test_idx] = pred

        valid = np.isfinite(oof_prob)
        tn, fp, fn, tp = confusion_matrix(y[valid], oof_pred[valid], labels=[0, 1]).ravel()
        rows.append({
            "schema": "lateralized_v1",
            "comparison": comparison_name,
            "feature_family": feature_family,
            "model": model_name,
            "n_subjects": int(valid.sum()),
            "n_features": len(feature_cols),
            "roc_auc": roc_auc_score(y[valid], oof_prob[valid]),
            "balanced_accuracy": balanced_accuracy_score(y[valid], oof_pred[valid]),
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "f1": f1_score(y[valid], oof_pred[valid]),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        })

        pred_df = pd.DataFrame({
            "participant_id": subset["participant_id"].to_numpy()[valid],
            "diagnosis": subset["diagnosis"].to_numpy()[valid],
            "y_true": y[valid],
            "y_prob": oof_prob[valid],
            "y_pred": oof_pred[valid].astype(int),
            "comparison": comparison_name,
            "feature_family": feature_family,
            "model": model_name,
            "schema": "lateralized_v1",
        })
        pred_df.to_csv(
            RESULTS_DIR / f"oof_predictions_lateralized_v1_{comparison_name}_{feature_family}_{model_name}.csv",
            index=False,
        )
    return pd.DataFrame(rows)


classification_results_lateralized_v1 = []
if RUN_LATERALIZED_CLASSIFICATION and not subject_features_lateralized_v1_full.empty:
    for comparison_name, groups in CLASSIFICATION_COMPARISONS.items():
        for family in [
            "psd",
            "hcfim",
            "psd_plus_hcfim",
            "lateral_deltas",
            "psd_plus_hcfim_plus_lateral_deltas",
        ]:
            result = evaluate_feature_family(subject_features_lateralized_v1_full, comparison_name, groups, family)
            if not result.empty:
                classification_results_lateralized_v1.append(result)
    classification_results_lateralized_v1 = pd.concat(classification_results_lateralized_v1, ignore_index=True)
    classification_results_lateralized_v1.to_csv(RESULTS_DIR / "classification_results_lateralized_v1.csv", index=False)
else:
    results_path = RESULTS_DIR / "classification_results_lateralized_v1.csv"
    classification_results_lateralized_v1 = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()

(
    classification_results_lateralized_v1
    .sort_values(["comparison", "roc_auc"], ascending=[True, False])
    .head(20)
    if not classification_results_lateralized_v1.empty
    else "Lateralized classification not run yet. Set RUN_FULL_LATERALIZED_FEATURE_EXTRACTION and RUN_LATERALIZED_CLASSIFICATION to True."
)

## 10. Classification Figures

Curves and confusion matrices are generated from out-of-fold predictions
for the best lateralized model per comparison.

In [ ]:
def plot_roc_and_confusion_for_best_lateralized(results: pd.DataFrame):
    if results.empty:
        print("No lateralized classification results available.")
        return []

    saved = []
    best_rows = (
        results
        .sort_values(["comparison", "roc_auc"], ascending=[True, False])
        .groupby("comparison", as_index=False)
        .head(1)
    )
    for row in best_rows.itertuples(index=False):
        pred_path = RESULTS_DIR / f"oof_predictions_lateralized_v1_{row.comparison}_{row.feature_family}_{row.model}.csv"
        if not pred_path.exists():
            continue
        pred = pd.read_csv(pred_path)
        fpr, tpr, _ = roc_curve(pred["y_true"], pred["y_prob"])
        cm = confusion_matrix(pred["y_true"], pred["y_pred"], labels=[0, 1]).astype(int)
        row_tot = cm.sum(axis=1, keepdims=True)
        cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100.0

        annot = np.array([[f"{int(cm[i, j])}/{int(row_tot[i, 0])}\n({cm_pct[i, j]:.1f}%)"
                   for j in range(2)] for i in range(2)])

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
        axes[0].plot(fpr, tpr, color="#2F5597", linewidth=2.2, label=f"AUC = {row.roc_auc:.3f}")
        axes[0].plot([0, 1], [0, 1], linestyle="--", color="#999999", linewidth=1.0)
        axes[0].set_xlabel("False positive rate")
        axes[0].set_ylabel("True positive rate")
        axes[0].legend(loc="lower right")

        sns.heatmap(
            cm_pct,
            annot=annot,
            fmt="",
            cmap="Blues",
            cbar=False,
            xticklabels=["control", "pathology"],
            yticklabels=["control", "pathology"],
            ax=axes[1],
        )
        axes[1].set_xlabel("Predicted")
        axes[1].set_ylabel("Observed")
        axes[1].set_title(f"Threshold = 0.50 | n = {int(cm.sum())}")

        output_path = FIGURES_DIR / f"classification_best_lateralized_v1_{row.comparison}.png"
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.show()
        saved.append(output_path)
    return saved


classification_lateralized_figure_paths = plot_roc_and_confusion_for_best_lateralized(
    classification_results_lateralized_v1
)
classification_lateralized_figure_paths

## 11. Compare Regional V1 Versus Lateralized V1

This comparison loads baseline regional_v1 results produced by notebook
01 when available. It is descriptive and intended to support the paper
decision on whether lateralization belongs in the main results or appendix.

In [ ]:
def load_regional_baseline_results() -> pd.DataFrame:
    path = RESULTS_DIR / "classification_results.csv"
    if not path.exists():
        return pd.DataFrame()
    baseline = pd.read_csv(path)
    baseline["schema"] = "regional_v1"
    return baseline

regional_results = load_regional_baseline_results()
combined_results = pd.concat(
    [regional_results, classification_results_lateralized_v1],
    ignore_index=True,
) if not classification_results_lateralized_v1.empty else regional_results

if not combined_results.empty:
    best_by_schema = (
        combined_results
        .sort_values(["comparison", "schema", "roc_auc"], ascending=[True, True, False])
        .groupby(["comparison", "schema"], as_index=False)
        .head(1)
        .sort_values(["comparison", "roc_auc"], ascending=[True, False])
    )
    display(best_by_schema)

    fig, ax = plt.subplots(figsize=(9, 4.8))
    sns.barplot(
        data=best_by_schema,
        x="comparison",
        y="roc_auc",
        hue="schema",
        palette={"regional_v1": "#4C78A8", "lateralized_v1": "#E45756"},
        ax=ax,
    )
    ax.set_xlabel("")
    ax.set_ylabel("ROC AUC")
    ax.set_ylim(0.45, 1.0)
    ax.legend(title="")
    ax.tick_params(axis="x", rotation=15)
    output_path = FIGURES_DIR / "regional_vs_lateralized_v1_best_auc.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("No regional or lateralized classifier results available yet.")

## 12. Takeaways Template

Complete after running the full lateralized extraction and classification.

Suggested reporting fields:

- Best lateralized HCFIM geometry for Control vs Alzheimer.
- Best lateralized HCFIM geometry for Control vs FTD.
- Whether lateralization improves ROC AUC, balanced accuracy, sensitivity
  or specificity over regional_v1.
- Whether right-minus-left deltas add signal beyond direct regional features.
- Whether the strongest separability appears in frontal, temporal,
  parietal, occipital or central regions.

Methodological caution:

- Lateralized_v1 increases the feature space in a small dataset.
- Interpret improvements only if they are stable across folds and not
  driven by one model family.
- If performance improves but geometric separation is weak, report the
  finding as predictive rather than geometric.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

data = [
    {
        "comparison": "Control vs AD",
        "schema": "Regional",
        "model": "PSD + HCFIM / Extra Trees",
        "roc_auc": 0.818,
        "ci_low": 0.696,
        "ci_high": 0.925,
    },
    {
        "comparison": "Control vs AD",
        "schema": "Lateralized v1",
        "model": "PSD + HCFIM + lateral deltas / Extra Trees",
        "roc_auc": 0.832,
        "ci_low": 0.721,
        "ci_high": 0.927,
    },
    {
        "comparison": "Control vs FTD",
        "schema": "Regional",
        "model": "HCFIM / MLP small",
        "roc_auc": 0.810,
        "ci_low": 0.677,
        "ci_high": 0.926,
    },
    {
        "comparison": "Control vs FTD",
        "schema": "Regional",
        "model": "PSD + HCFIM / Extra Trees",
        "roc_auc": 0.775,
        "ci_low": 0.634,
        "ci_high": 0.901,
    },
    {
        "comparison": "Control vs FTD",
        "schema": "Lateralized v1",
        "model": "PSD + HCFIM / Logistic ElasticNet",
        "roc_auc": 0.741,
        "ci_low": 0.592,
        "ci_high": 0.865,
    },
]

df_auc = pd.DataFrame(data)

best_auc = (
    df_auc
    .sort_values(["comparison", "schema", "roc_auc"], ascending=[True, True, False])
    .groupby(["comparison", "schema"], as_index=False)
    .head(1)
)

best_auc["err_low"] = best_auc["roc_auc"] - best_auc["ci_low"]
best_auc["err_high"] = best_auc["ci_high"] - best_auc["roc_auc"]

fig, ax = plt.subplots(figsize=(8.5, 4))

sns.barplot(
    data=best_auc,
    x="comparison",
    y="roc_auc",
    hue="schema",
    hue_order=["Regional", "Lateralized v1"],
    palette={ "Regional": "#084081", "Lateralized v1": "#5cbbce" },
    errorbar=None,
    ax=ax,
)

for patch, (_, row) in zip(ax.patches, best_auc.iterrows()):
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()

    ax.errorbar(
        x=x,
        y=y,
        yerr=[[row["err_low"]], [row["err_high"]]],
        fmt="none",
        ecolor="#333333",
        elinewidth=1.2,
        capsize=4,
        capthick=1.2,
        zorder=5,
    )

    ax.text(
        x,
        y + row["err_high"] + 0.012,
        f"{y:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        color="#222222",
    )

ax.set_xlabel("")
ax.set_ylabel("ROC AUC")
ax.set_ylim(0.45, 1.0)
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

ax.legend(title="", frameon=False, loc="upper right")
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=0)

sns.despine(ax=ax)

output_path = FIGURES_DIR / "regional_vs_lateralized_v1_best_auc.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.15)

plt.show()